In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os

PROJECT_PATH = "/content/drive/MyDrive/AQI_Seasonal_Prediction"

DATASET_PATH = os.path.join(PROJECT_PATH, "datasets")
RESULT_PATH = os.path.join(PROJECT_PATH, "results")
FIGURE_PATH = os.path.join(PROJECT_PATH, "figures")
MODEL_PATH = os.path.join(PROJECT_PATH, "models")

In [3]:
merged_full = pd.read_csv(
    os.path.join(DATASET_PATH, "merged_allcolumns_aqi_weather_2016_2022.csv")
)

print(merged_full.shape)
merged_full.head()

(53355, 31)


,Date (LT),Hour,NowCast Conc.,Raw Conc.,Conc. Unit,AQI,AQI Category,QC Name,temperature_2m,relativehumidity_2m,...,windgusts_10m,et0_fao_evapotranspiration,vapor_pressure_deficit,shortwave_radiation,direct_radiation,diffuse_radiation,weathercode,visibility,soil_temperature_0cm,soil_moisture_0_1cm
0,2016-01-01 01:00:00,1,-999.0,-999,ug/m3,-999,NaN,Missing,14.0,95,...,8.6,0.00,0.08,3.0,0.0,3.0,2,NaN,NaN,NaN
1,2016-01-01 02:00:00,2,-999.0,-999,ug/m3,-999,NaN,Missing,15.9,93,...,10.1,0.04,0.12,105.0,59.0,46.0,2,NaN,NaN,NaN
2,2016-01-01 03:00:00,3,-999.0,-999,ug/m3,-999,NaN,Missing,19.9,76,...,11.5,0.16,0.56,291.0,205.0,86.0,0,NaN,NaN,NaN
3,2016-01-01 04:00:00,4,-999.0,-999,ug/m3,-999,NaN,Missing,22.4,65,...,13.7,0.29,0.94,464.0,352.0,112.0,0,NaN,NaN,NaN
4,2016-01-01 05:00:00,5,-999.0,-999,ug/m3,-999,NaN,Missing,24.0,57,...,16.2,0.40,1.29,609.0,494.0,115.0,0,NaN,NaN,NaN


In [ ]:
#datacleaning
print("Shape:", merged_full.shape)
merged_full.info()
merged_full.head()

In [25]:
#missing values
missing = merged_full.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

missing



#visibility,soil_temperature_0cm,soil_moisture_0_1cm contains no data. So dropping these columns
# merged_full.drop(
#     columns=[
#         "visibility",
#         "soil_temperature_0cm",
#         "soil_moisture_0_1cm"
#     ],
#     inplace=True
# )

# print(merged_full.shape)

#missing percentage after dropping
missing_percent = (merged_full.isnull().sum() / len(merged_full)) * 100

missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)

print(missing_percent)

#Check the AQI quality flag
print(merged_full["QC Name"].value_counts(dropna=False))

#crosscheck
pd.crosstab(
    merged_full["QC Name"],
    merged_full["AQI Category"].isna()
)

merged_full[
    (merged_full["QC Name"] == "Valid") &
    (merged_full["AQI Category"].isna())
][["Date (LT)", "AQI", "Raw Conc.", "NowCast Conc."]].head(20)

print("AQI = -999:", (merged_full["AQI"] == -999).sum())
print("Raw Conc. = -999:", (merged_full["Raw Conc."] == -999).sum())
print("NowCast Conc. = -999:", (merged_full["NowCast Conc."] == -999).sum())

invalid_mask = (
    (merged_full["AQI"] == -999) |
    (merged_full["Raw Conc."] == -999) |
    (merged_full["NowCast Conc."] == -999)
)

print("Rows with at least one -999:", invalid_mask.sum())

all_invalid = (
    (merged_full["AQI"] == -999) &
    (merged_full["Raw Conc."] == -999) &
    (merged_full["NowCast Conc."] == -999)
)

print("Rows with all three = -999:", all_invalid.sum())

print(merged_full["AQI Category"].unique())



#as model will predict aqi category so removing those with null values of aqi category
cleaned = merged_full.dropna(
    subset=["AQI Category"]
)

print(cleaned.shape)


#Remove AQI invalid sentinel rows
invalid_aqi = (
    (cleaned["AQI"] == -999) |
    (cleaned["Raw Conc."] == -999) |
    (cleaned["NowCast Conc."] == -999)
)

cleaned = cleaned[~invalid_aqi]

print(cleaned.shape)

cleaned["AQI Category"].value_counts()

#checking missing
missing = (
    cleaned.isnull()
    .sum()
    /
    len(cleaned)
)*100

missing[missing>0]

cleaned.isnull().sum()

,0
Date (LT),0
Hour,0
NowCast Conc.,0
Raw Conc.,0
Conc. Unit,0
AQI,0
AQI Category,0
QC Name,0
temperature_2m,0
relativehumidity_2m,0


In [46]:
#checking duplicates
cleaned["Date (LT)"].duplicated().sum()

#show duplicates
duplicates = cleaned[
    cleaned["Date (LT)"].duplicated(keep=False)
]

duplicates.sort_values("Date (LT)")


#identify the duplicates
duplicate_times = cleaned[
    cleaned["Date (LT)"].duplicated(keep=False)
]["Date (LT)"].unique()

duplicate_times

#Separate duplicate and non-duplicate rows
duplicates = cleaned[
    cleaned["Date (LT)"].isin(duplicate_times)
]

non_duplicates = cleaned[
    ~cleaned["Date (LT)"].isin(duplicate_times)
]
print("Original shape:", cleaned.shape)
print("Duplicates shape:", duplicates.shape)
print("Non-duplicates shape:", non_duplicates.shape)

#Aggregate only the duplicate rows
duplicate_avg = duplicates.groupby("Date (LT)", as_index=False).agg(
    {
        "NowCast Conc.": "mean",
        "Raw Conc.": "mean",
        "AQI": "mean",

        "Hour": "first",
        "Conc. Unit": "first",
        "QC Name": "first",

        "temperature_2m": "first",
        "relativehumidity_2m": "first",
        "dewpoint_2m": "first",
        "apparent_temperature": "first",
        "precipitation": "first",
        "rain": "first",
        "surface_pressure": "first",
        "cloudcover": "first",
        "cloudcover_low": "first",
        "cloudcover_mid": "first",
        "cloudcover_high": "first",
        "windspeed_10m": "first",
        "winddirection_10m": "first",
        "windgusts_10m": "first",
        "et0_fao_evapotranspiration": "first",
        "vapor_pressure_deficit": "first",
        "shortwave_radiation": "first",
        "direct_radiation": "first",
        "diffuse_radiation": "first",
        "weathercode": "first"
    }
)

print(duplicate_avg.shape)

#give the aqicategory for the mean values
def get_aqi_category(aqi):
    if aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Moderate"
    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups"
    elif aqi <= 200:
        return "Unhealthy"
    elif aqi <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

duplicate_avg["AQI Category"] = duplicate_avg["AQI"].apply(get_aqi_category)

duplicate_avg[["Date (LT)", "AQI", "AQI Category"]]

#add this with non_cuplicates
cleaned = pd.concat(
    [non_duplicates, duplicate_avg],
    ignore_index=True
)

#sort by time
cleaned = cleaned.sort_values("Date (LT)").reset_index(drop=True)

#verification
print("Shape:", cleaned.shape)

print("Duplicate timestamps:",
      cleaned["Date (LT)"].duplicated().sum())

print("Missing values:")
print(cleaned.isnull().sum()[cleaned.isnull().sum() > 0])


Shape: (51031, 28)
Duplicate timestamps: 0
Missing values:
Series([], dtype: int64)


In [48]:
#save

cleaned.to_csv(
    os.path.join(DATASET_PATH, "cleaned_allmerged_aqi_weather.csv"),
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [50]:
print(cleaned.info())

print("\nAQI Category distribution:")
print(cleaned["AQI Category"].value_counts())

print("\nDate range:")
print(cleaned["Date (LT)"].min())
print(cleaned["Date (LT)"].max())


print(cleaned["Date (LT)"].min())
print(cleaned["Date (LT)"].max())

print(merged_full["Date (LT)"].min())
print(merged_full["Date (LT)"].max())

print("Original merged:", len(merged_full))
print("After cleaning:", len(cleaned))
print("Rows removed:", len(merged_full) - len(cleaned))

2016-03-01 03:00:00
2022-06-01 01:00:00
2016-01-01 01:00:00
2022-06-01 01:00:00
Original merged: 53355
After cleaning: 51031
Rows removed: 2324


In [ ]:
#After removing records with missing target labels (AQI Category) and invalid sentinel values (-999), the earliest valid observation begins on 2016-03-01. This indicates that the January–February 2016 records in the original dataset were not suitable for model training.